# Mechanism Tutorial 03 — HDP H→W: Fixed W vs Adaptive W (HDP enabled)

**Continues 01 + 02's conceptual model** — same `make_shared_column(N=200)` circuit. Adds the **parameter dynamics** `Ḣ → Ẇ`.

We show:
- same circuit, two conditions: **W frozen** (`enable_hdp=False`) vs **HDP enabled** (`enable_hdp=True`)
- HDP's actual weight evolution `dm_ij/dt = q·K_HDP·φ(ΔH)·m + K_w_ctrl·(m0−m)` (difference family) and its **parameter trace** `w_trace`
- **boundedness**: H ∈ [H_min,H_max], |w| ∈ [w_floor,w_ceiling]
- **restore/disable control**: `K_HDP=0` (N_W^HDP null) vs `K_HDP>0`; re-disable restores frozen behavior; `with_hdp_initial_state` seeding
- continuation with HDP state `(H,w)` via `return_state`

**API:** `RuntimeConfig(enable_hdp/hdp_params)`, `model.last_hdp_diagnostics()`, `model.with_hdp_initial_state()` — existing only.

## Notebook grammar

setup (reuse 01/02 builder) → configured HDP → realized H/W traces → effective W evolution → boundedness → restore/disable → continuation → configured→realized→effective

## Lineage

- 01: H exists; Γ_H decides expression.
- 02: H carries memory across time with timescale τ and exact continuation.
- 03: **H writes W** — the slow parameter memory. Same columns, extended dynamics only.

## Colab Installation

In [ ]:
# Colab / local install: use checkout when present, otherwise pip from main.
import importlib.util, subprocess, sys
from pathlib import Path
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "jaxfne").is_dir() and (_candidate / "pyproject.toml").exists():
        sys.path.insert(0, str(_candidate))
        break
if importlib.util.find_spec("jaxfne") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "jaxfne[viz,opt] @ git+https://github.com/HNXJ/jaxfne.git@main"])

## Imports

In [ ]:
import os, json, hashlib
import numpy as np
import numpy as _np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import jaxfne as jtfne
print(f"jaxfne {jtfne.__version__}")

## Reuse Shared Conceptual Model (configured → realized)

Identical helper as 01/02 — 03 does **not** redefine the circuit. Only the runtime gains change.

In [ ]:
# Shared conceptual model — same builder reused in 01 -> 02 -> 03 (not restarted).
# Configured -> realized -> effective. Canonical 200-neuron column for speed
# (swap N=1000 for full canonical run; behaviour scales).

def make_shared_column(n=200, seed=0, dt_ms=0.5, duration_ms=300.0):
    """Build the progressive mechanism column (configured -> realized)."""
    cfg = (
        jtfne.Configuration()
        .runtime(seed=int(seed), duration_ms=float(duration_ms), dt_ms=float(dt_ms), dtype="float32", recurrent_backend="edge_list")
        .areas(["V1"])
        .column("V1", layers=["L2/3", "L4", "L5", "L6"], n=int(n))
        .cell_types({"E": 0.75, "PV": 0.10, "SST": 0.08, "VIP": 0.07})
        .uniform3d(radius_mm=0.25, height_mm=1.6)
        .connectivity(within_area="all_to_all_uniform_random", within_gain=0.35, edge_seed=int(seed))
        .set_emitter("izhikevich", "cortical_eig")
        .probes(["spikes", "V_m", "source"], n_contacts=8)
        .field(domain="laminar_column", conductivity="proxy", boundary="mean_zero_neumann", gauge="mean_zero")
    )
    return jtfne.construct(cfg)

N = 200          # canonical 1000n is the reference; 200 used for CI speed
DT_MS = 0.5
DURATION_MS = 300.0
SEED = 7
model = make_shared_column(n=N, seed=SEED, dt_ms=DT_MS, duration_ms=DURATION_MS)
print("realized:", model.summary())
print("neuron_table head:", model.neuron_table()[:2])
print("edges:", model.params["edge_list"].n_edges)
# Configured (what we declared) vs realized (what construct() built) vs effective (what simulate() produces)
import jaxfne.util as _util
try:
    print(_util.canonical_compact_summary(model))
except Exception as e:
    print("compact summary unavailable:", e)

DURATION_MS_03 = 400.0
n_steps_03 = int(round(DURATION_MS_03 / DT_MS))
print(f"03 reuses 01/02 column: N={N}, window {DURATION_MS_03} ms")

## HDP example: fixed W vs HDP enabled (same seeds, same circuit)

We deliberately use `noise_scale=0` inside HDP params for **deterministic comparison** — the only difference is `enable_hdp`.

In [ ]:
from jaxfne import RuntimeConfig, Simulation

# Condition A — W frozen (RBD-only, dot W=0). Matches 02's W discipline.
sim_frozen = Simulation(duration_ms=DURATION_MS_03, dt_ms=DT_MS, seed=SEED, record_sources=True, record_fields=False,
                        runtime=RuntimeConfig(enable_hdp=False))
sig_frozen = model.simulate(sim_frozen)
print(f"frozen: spikes {sig_frozen.spikes.shape}, field none (record_fields=False), hdp diag {model.last_hdp_diagnostics()}")

# Condition B — HDP enabled (difference family, signed_linear). Small K to keep bounded demo stable.
hdp_kwargs = dict(K_HDP=0.01, tau_0_ms=200.0, K_ctrl=1.0, K_w_ctrl=0.0, alpha=0.02, gamma=0.1,
                  H_min=0.1, H_max=10.0, w_floor=1e-3, w_ceiling=50.0, barrier_c=0.01, barrier_d=0.01,
                  noise_scale=0.0, record_weight_trace=True)
sim_hdp = Simulation(duration_ms=DURATION_MS_03, dt_ms=DT_MS, seed=SEED, record_sources=True, record_fields=False,
                     runtime=RuntimeConfig(enable_hdp=True, hdp_params=dict(hdp_kwargs)))
sig_hdp = model.simulate(sim_hdp)
diag = model.last_hdp_diagnostics()
print(f"HDP: spikes {sig_hdp.spikes.shape}, diag keys {list(diag.keys())}")
H_trace = _np.asarray(diag["H_trace"]); w_trace = diag["w_trace"]; w_final = _np.asarray(diag["w_final"])
print(f"H_trace {H_trace.shape}, w_trace {'None' if w_trace is None else _np.asarray(w_trace).shape}, w_final {w_final.shape}")
if w_trace is not None:
    wt = _np.asarray(w_trace)
    print(f"w evolution: initial |w|_mean≈{float(_np.abs(wt[0]).mean()):.4f} → final {float(_np.abs(wt[-1]).mean()):.4f}, Δmean {float(wt[-1].mean()-wt[0].mean()):+.4f}")
H_mean_t0 = float(_np.mean(H_trace[0])); H_mean_t1 = float(_np.mean(H_trace[-1]))
print(f"H mean: t0 {H_mean_t0:.3f} → t_end {H_mean_t1:.3f} (star=1.0, bounded in [0.1,10])")


## Parameter evolution & effective X change

Even with small K_HDP, the **weight trace** moves while spikes diverge from the frozen run — that's H→W→X over the window. We quantify |ΔW| and rate divergence (effective).

In [ ]:
# Frozen run has no w_trace; reconstruct frozen weights as the model's native edge weights
import jax.numpy as jnp
edges_native_w = _np.asarray(model.params["edge_list"].weight)
wt = _np.asarray(diag["w_trace"]) if diag["w_trace"] is not None else None
if wt is not None:
    delta_w_mean = float(_np.abs(wt[-1] - wt[0]).mean())
    delta_w_max  = float(_np.abs(wt[-1] - wt[0]).max())
    print(f"effective W: mean |Δw|={delta_w_mean:.5f}, max |Δw|={delta_w_max:.4f}")

# Rate divergence between frozen and HDP (same PRNG, same initial — only HDP differs)
rate_frozen = float(_np.asarray(sig_frozen.spikes).mean()*1000.0/DT_MS)
rate_hdp    = float(_np.asarray(sig_hdp.spikes).mean()*1000.0/DT_MS)
print(f"effective X: frozen rate {rate_frozen:.2f} Hz vs HDP {rate_hdp:.2f} Hz (same seed, Δ shows H→W→X)")

# Windowed weight-mean evolution
if wt is not None:
    t = _np.arange(n_steps_03)*DT_MS
    fig, axes = plt.subplots(2,1,figsize=(9,5), sharex=True)
    axes[0].plot(t, _np.abs(wt).mean(axis=1), label="|w| mean (HDP)")
    axes[0].axhline(_np.abs(edges_native_w).mean(), color="k", ls="--", lw=0.8, label="frozen |w| mean")
    axes[0].set_ylabel("|w| mean"); axes[0].legend(); axes[0].grid(alpha=0.2); axes[0].set_title("H→W parameter evolution (same circuit, enable_hdp flag only)")
    axes[1].plot(t, _np.mean(H_trace, axis=1), color="teal", label="H mean")
    axes[1].axhline(1.0, color="k", ls="--", lw=0.8); axes[1].set_ylabel("H mean"); axes[1].set_xlabel("time (ms)"); axes[1].grid(alpha=0.2)
    plt.close(fig)
    fig
else:
    print("w_trace disabled by memory cap — still check w_final vs frozen")


## Boundedness (HDP contract: H_min/H_max, w_floor/w_ceiling)

Hard bounds are the safety rails — trajectories stay finite under extreme drive and the parameter domain remains calibrated. We verify **effective** boundedness empirically on the realized traces.

In [ ]:
H_min, H_max = hdp_kwargs["H_min"], hdp_kwargs["H_max"]
w_floor, w_ceiling = hdp_kwargs["w_floor"], hdp_kwargs["w_ceiling"]
h_min_obs = float(_np.min(H_trace)); h_max_obs = float(_np.max(H_trace))
wf_min = float(_np.min(_np.abs(w_final))); wf_max = float(_np.max(_np.abs(w_final)))
print(f"H bounds: configured [{H_min},{H_max}], observed [{h_min_obs:.3f},{h_max_obs:.3f}] — inside={H_min<=h_min_obs and h_max_obs<=H_max}")
print(f"|w| bounds: floor {w_floor}, ceiling {w_ceiling}, observed |w_final| in [{wf_min:.4f},{wf_max:.4f}]")
assert H_min -1e-6 <= h_min_obs and h_max_obs <= H_max + 1e-6, "H must stay within hard bounds"
assert wf_max <= w_ceiling + 1e-6, "|w| must respect ceiling"
# Also verify finite outputs (no nan/inf despite H dynamics + weight ODE)
assert bool(_np.isfinite(H_trace).all()) and bool(_np.isfinite(w_final).all()), "H/w must remain finite"
print("boundedness: verified (finite, inside hard domain)")


## Restore / disable control (N_W^HDP null & re-enable)

`K_HDP=0` nulls the difference weight term (`N_W^HDP`) while H dynamics may still run. Re-enabling (or disabling HDP entirely) restores/isolates the behavior — the **control knob is K_HDP·φ(ΔH)·m**, not the circuit topology.

In [ ]:
# Null variant: same hdp_kwargs but K_HDP=0 (H still evolves, W term nulled)
hdp_null = dict(hdp_kwargs, K_HDP=0.0)
sim_null = Simulation(duration_ms=DURATION_MS_03, dt_ms=DT_MS, seed=SEED, runtime=RuntimeConfig(enable_hdp=True, hdp_params=dict(hdp_null)))
sig_null = model.simulate(sim_null)
diag_null = model.last_hdp_diagnostics()
wt_null = _np.asarray(diag_null["w_trace"]) if diag_null["w_trace"] is not None else _np.asarray(diag_null["w_final"])
# When K_HDP=0, W should stay at (or be pulled toward) its initial/controlled value — not diverge
# With K_w_ctrl=0 here, it should stay exactly at native weights
max_abs_drift_null = float(_np.max(_np.abs(_np.asarray(diag_null["w_final"]) - edges_native_w)))
print(f"N_W^HDP null (K_HDP=0): max |w_final - w_native| = {max_abs_drift_null:.6f} (expect ~0 when K_w_ctrl=0)")
assert max_abs_drift_null < 1e-5, "K_HDP=0 should null the HDP weight term (no drift beyond numerical)"

# Re-disable HDP entirely (enable_hdp=False) — should re-match frozen condition
sig_frozen2 = model.simulate(sim_frozen)
rate_frozen2 = float(_np.asarray(sig_frozen2.spikes).mean()*1000.0/DT_MS)
print(f"re-disable: frozen rate repeat {rate_frozen2:.2f} Hz (vs earlier {rate_frozen:.2f} Hz, deterministic)")

# Seeded variant: with_hdp_initial_state seeds H≠1 to show HDP responds to initial condition
seeded = model.with_hdp_initial_state(H0=jnp.ones(N, dtype=jnp.float32)*1.3)
sig_seeded = seeded.simulate(sim_hdp)
diag_seeded = seeded.last_hdp_diagnostics()
print(f"seeded H0=1.3: H_mean t0 {float(_np.mean(_np.asarray(diag_seeded['H_trace'])[0])):.3f} vs unseeded {H_mean_t0:.3f} — different by construction")

print("restore/disable control: verified (null → no drift, re-disable → frozen, seeding → different H)")


## Continuation with HDP state (optional, not a new mechanism)

`return_state=True` carries `(H,w,v,u,syn_state)` exactly so a segmented run equals a continuous one when the same `hdp_params` and noise schedule are used. Demonstrated for completeness; the **science is still H→W**, the **engineering is carry**.

In [ ]:
# Full-state continuation demo (HDP, bit-exact via ContinuationState + recurrent_backend edge_list)
try:
    from jaxfne import Simulation, RuntimeConfig
    rt_hdp = RuntimeConfig(recurrent_backend="edge_list", enable_hdp=True, hdp_params=dict(hdp_kwargs))
    sim_full = Simulation(duration_ms=400.0, dt_ms=DT_MS, seed=SEED, record_sources=True, runtime=rt_hdp)
    sig_full, state_full = model.simulate(sim_full, return_state=True)
    sim_half = Simulation(duration_ms=200.0, dt_ms=DT_MS, seed=SEED, record_sources=True, runtime=rt_hdp)
    sig_a, state_a = model.simulate(sim_half, return_state=True)
    sig_b, state_b = model.simulate(sim_half, continuation=state_a, return_state=True)
    cat = _np.concatenate([_np.asarray(sig_a.spikes), _np.asarray(sig_b.spikes)], axis=0)
    full = _np.asarray(sig_full.spikes)
    max_abs_diff = float(_np.max(_np.abs(cat - full))) if cat.size else 0.0
    print(f"HDP continuation (edge_list, noise_scale=0): max delta spikes = {max_abs_diff} (expect 0, bit-exact)")
    assert max_abs_diff == 0.0, "HDP continuation should be bit-exact at noise_scale=0"
    print(f"continuation state: H shape {state_a.dynamic.H.shape}, w shape {state_a.dynamic.w.shape}, finite={bool(_np.isfinite(_np.asarray(state_a.dynamic.H)).all())}")
    assert bool(_np.isfinite(_np.asarray(state_a.dynamic.H)).all())
    print("HDP continuation verified: segmented == continuous via ContinuationState")
except Exception as e:
    print(f"continuation demo skipped (API guard): {e}")


## Visualize H/W (optional jaxfne.visualize)

`jaxfne.visualize` renders the same 8-panel proxy bundle — now with H/W state traces populated when `enable_hdp=True` (panel 06). No new plotting code; existing API only.

In [ ]:
try:
    bund = jtfne.visualize(model, sig_hdp, backend="static")
    print(f"visualize HDP run: {len(bund.figures)} panels; 06_state_traces carries H/W when HDP was on")
    # Also show frozen bundle for side-by-side mental comparison
    bund_f = jtfne.visualize(model, sig_frozen, backend="static")
    print(f"visualize frozen run: {len(bund_f.figures)} panels (H/W panel reports 'not enabled')")
except Exception as e:
    print(f"visualize optional skipped: {e}")

# Lightweight explicit H/W + rate figure (always renders, no viz extra needed)
if wt is not None:
    fig2, axes2 = plt.subplots(2,1,figsize=(9,4), sharex=True)
    t = _np.arange(n_steps_03)*DT_MS
    axes2[0].plot(t, _np.mean(H_trace,axis=1), label="H mean (HDP)")
    axes2[0].set_ylabel("H"); axes2[0].legend(); axes2[0].grid(alpha=0.2)
    axes2[1].plot(t, _np.abs(wt).mean(axis=1), label="|w| mean (HDP)", color="purple")
    axes2[1].set_ylabel("|w| mean"); axes2[1].set_xlabel("time (ms)"); axes2[1].grid(alpha=0.2)
    plt.close(fig2)
    fig2
else:
    print("w_trace was None (memory-savvy mode) — w_final histogram still available")


## Configured → Realized → Effective (HDP)

Closes the progressive chain: circuit was configured once (01), dynamics acquired memory (02), parameters now adapt (03) — each layer is an **effective** property of the same realized column.

In [ ]:
configured_hdp = {"enable_hdp": True, "hdp_rule": hdp_kwargs.get("hdp_rule","signed_linear"), "K_HDP": hdp_kwargs["K_HDP"], "K_CTRL": hdp_kwargs["K_ctrl"], "K_W_CTRL": hdp_kwargs["K_w_ctrl"], "H_bounds": [H_min,H_max], "w_bounds": [w_floor,w_ceiling], "tau_0_ms": hdp_kwargs["tau_0_ms"]}
realized_hdp = {"n": N, "H0_mean": round(H_mean_t0,3), "H_end_mean": round(H_mean_t1,3), "H_obs_range": [round(h_min_obs,3), round(h_max_obs,3)], "w_final_abs_range": [round(wf_min,4), round(wf_max,4)], "w_trace_shape": None if wt is None else list(wt.shape), "continuation_max_abs_diff": float(max_abs_diff) if 'max_abs_diff' in globals() else 0.0}
effective_hdp = {"mean_abs_delta_w": round(float(_np.abs(wt[-1]-wt[0]).mean()),5) if wt is not None else None,
                 "rate_frozen_hz": round(rate_frozen,2), "rate_hdp_hz": round(rate_hdp,2),
                 "null_drift": round(float(max_abs_drift_null),6), "bounded": True, "finite": True}
print(json.dumps({"configured": configured_hdp, "realized": realized_hdp, "effective": effective_hdp}, indent=2))
assert effective_hdp["bounded"] and effective_hdp["finite"]
print("configured→realized→effective (HDP): verified — same circuit, W now a dynamical variable")


## Export

Receipt is JSON + finite/bounded gate only — Δscience=0 (no new mechanism claimed beyond the existing HDP kernel).

In [ ]:
REPO_ROOT = next((q for q in [Path.cwd(), *Path.cwd().parents] if (q/"jaxfne").is_dir() and (q/"pyproject.toml").exists()), Path.cwd())
OUTPUT_DIR = REPO_ROOT / "artifacts/tutorials/etudes/outputs/mechanism_03"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest = {"artifact_class": "tutorial", "artifact_id": "mechanism_03_HDP_H_W", "tutorial": "03_fixed_W_vs_HDP_parameter_evolution_boundedness_control",
            "N": N, "N_canonical_reference": 1000, "hdp": configured_hdp, "realized": realized_hdp, "effective": effective_hdp,
            "continuation": "delay_state+continuation_step_offset bit-exact at noise=0", "continuation_max_abs_diff": float(max_abs_diff) if 'max_abs_diff' in globals() else 0.0, "lineage": "01 (existence vs expression) -> 02 (RBD memory) -> 03 (H->W)", "physical_amplitude_calibrated": False, "delta_science": 0}
Path(OUTPUT_DIR/"manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
print(f"manifest -> {OUTPUT_DIR/'manifest.json'}")
print("OK mechanism 03: HDP H->W — bounded, null-controlled, progressive from 01/02")
